# 10 - CRISPR-Cas12a deletion example in *Aspergillus oryzae*


This notebook is a toned-down CRISPR design example inspired by a more elaborate StreptoCAD workflow. The goal here is not to reproduce a full plasmid construction workflow, but to show how `teemi` can be used to:

- load a genome record
- identify candidate Cas12a guides for selected genes
- choose a small subset of guides per target
- fetch repair oligos for a clean deletion strategy

To keep the example compact and readable, this notebook stops at design tables rather than building a full project export folder. For guide design and off-target screening we concatenate all eight *A. oryzae* RIB40 chromosomes into one combined genome record, while keeping the outputs dataframe-heavy.

This workflow uses a 23 nt Cas12a protospacer length.

The genome files are not bundled in the repository. Place `chrom1.gb` to `chrom8.gb` in `teemi_cad_workflows/inspiration_notebooks/data/09_aspergillus_crispr_example/` before running the notebook.


In [1]:
%%capture
import sys
if 'google.colab' in sys.modules:
    %pip install pydna teemi biopython pandas

## Imports

In [2]:
from pathlib import Path

import pandas as pd
from pydna.dseqrecord import Dseqrecord

from teemi.design.fetch_sequences import read_genbank_files, concatenate_genbank_records
from teemi.design.crispr_cas import SgRNAargs, extract_sgRNAs
from teemi.design.gibson_cloning import extract_locus_tag_homology_arms

## Input data

In [3]:
# This example uses all eight Aspergillus oryzae RIB40 chromosomes.
genome_dir = Path('../data/09_aspergillus_crispr_example')
chromosome_paths = sorted(genome_dir.glob('chrom*.gb'), key=lambda path: int(path.stem.replace('chrom', '')))

if len(chromosome_paths) != 8:
    raise FileNotFoundError(
        'Expected chrom1.gb to chrom8.gb in ../data/09_aspergillus_crispr_example/. '
        'Add the local Aspergillus genome files before running this notebook.'
    )

chromosome_records = [read_genbank_files(str(path))[0] for path in chromosome_paths]

chromosome_summary_df = pd.DataFrame([
    {
        'chromosome': record.id,
        'length_bp': len(record.seq),
        'feature_count': len(record.features),
    }
    for record in chromosome_records
])

chromosome_summary_df

,chromosome,length_bp,feature_count
0,BA000049.1,6520266,10184
1,BA000050.1,6264705,10054
2,BA000051.1,5123684,8291
3,BA000052.1,4887096,7596
4,BA000053.1,4533889,7378
5,BA000054.1,4190347,6562
6,BA000055.1,2933481,4571
7,BA000056.1,3395259,5397


In [4]:
combined_genome_record = concatenate_genbank_records(
    chromosome_records,
    record_id='A_oryzae_RIB40_combined',
    record_name='A_oryzae_RIB40_combined',
)
combined_genome = Dseqrecord(combined_genome_record)

pd.DataFrame([
    {
        'combined_record_id': combined_genome_record.id,
        'combined_length_bp': len(combined_genome_record.seq),
        'combined_feature_count': len(combined_genome_record.features),
    }
])

,combined_record_id,combined_length_bp,combined_feature_count
0,A_oryzae_RIB40_combined,37848727,60033


For a lightweight example, we use three early locus tags from chromosome 1. You can replace these with any other locus tags from the same GenBank file.

In [5]:
genes_to_delete = [
    'AO090009000001',
    'AO090009000002',
    'AO090009000003',
]

genes_to_delete

['AO090009000001', 'AO090009000002', 'AO090009000003']

## Find candidate Cas12a guides

In [6]:
cas12a_protospacer_len = 23

args = SgRNAargs(
    combined_genome,
    genes_to_delete,
    cas_type='cas12a',
    step=['find', 'filter'],
    gc_upper=0.72,
    gc_lower=0.35,
    off_target_seed=13,
    off_target_upper=5,
    protospacer_len=cas12a_protospacer_len,
)

sgrna_df = extract_sgRNAs(args)
sgrna_df.head()

,strain_name,locus_tag,gene_loc,gene_strand,sgrna_strand,sgrna_loc,gc,pam,sgrna,sgrna_seed_sequence,off_target_count
0,A_oryzae_RIB40_combined,AO090009000001,882,1,1,156,0.434783,TTTC,TCACATGGATAACAACAGAGGAG,TCACATGGATAAC,0
76,A_oryzae_RIB40_combined,AO090009000003,15692,-1,-1,534,0.521739,TTTA,TCCAGAGTCAGATTAACCAGGCG,TCCAGAGTCAGAT,0
75,A_oryzae_RIB40_combined,AO090009000003,15692,-1,-1,468,0.565217,TTTG,CCGTTATGCCAGCAAGCCAGACA,CCGTTATGCCAGC,0
74,A_oryzae_RIB40_combined,AO090009000003,15692,-1,-1,464,0.521739,TTTC,TTTGCCGTTATGCCAGCAAGCCA,TTTGCCGTTATGC,0
73,A_oryzae_RIB40_combined,AO090009000003,15692,-1,-1,231,0.565217,TTTC,CTTCACCAGACAGTCCCCCGATT,CTTCACCAGACAG,0


Here we keep the top two filtered Cas12a guides per locus tag, ordered by off-target count. The example below uses 23 nt protospacers.

In [7]:
selected_guides = (
    sgrna_df
    .sort_values(['locus_tag', 'off_target_count', 'gc'])
    .groupby('locus_tag')
    .head(2)
    .reset_index(drop=True)
)

selected_guides[['locus_tag', 'gene_loc', 'sgrna_loc', 'sgrna_strand', 'gc', 'pam', 'sgrna', 'off_target_count']]

,locus_tag,gene_loc,sgrna_loc,sgrna_strand,gc,pam,sgrna,off_target_count
0,AO090009000001,882,156,1,0.434783,TTTC,TCACATGGATAACAACAGAGGAG,0
1,AO090009000001,882,4129,-1,0.434783,TTTA,TGATCTTCATCTCCTGTCTCAGA,0
2,AO090009000002,9683,4131,-1,0.434783,TTTA,TACGATCCCTGAAGATCCAGTAT,0
3,AO090009000002,9683,851,-1,0.434783,TTTG,GAGTTTTGCATCGAGCTGATGAA,0
4,AO090009000003,15692,816,-1,0.391304,TTTG,ATCAAATCCAGATCAACACCTTC,0
5,AO090009000003,15692,726,1,0.391304,TTTG,AACAAAGACAGCTAGTAAGGGTT,0


## Fetch 45 bp homology arms for repair

For this example we keep the repair design minimal: a user-defined number of base pairs immediately upstream of the CDS plus the same number immediately downstream of the CDS. These are concatenated into one repair oligo for ordering.

In [8]:
repair_arm_length = 45

homology_arm_df = extract_locus_tag_homology_arms(
    combined_genome_record,
    genes_to_delete,
    arm_length=repair_arm_length,
)

homology_arm_df

,locus_tag,gene_start,gene_end,gene_strand,upstream_arm_name,downstream_arm_name,repair_oligo_name,upstream_arm_start,upstream_arm_end,downstream_arm_start,downstream_arm_end,upstream_arm_length,downstream_arm_length,repair_oligo_length,upstream_arm,downstream_arm,repair_oligo
0,AO090009000001,881,5162,1,AO090009000001_UP45,AO090009000001_DW45,AO090009000001_DEL_45bp_arms,836,881,5162,5207,45,45,90,TCCCCTTCTGGCTCTCCTCCCCCCCCCCCCCCGGAGGGCTGCGCC,TTTAGCTTATACAGTGTATATGAATATATATATTATATAGATTAA,TCCCCTTCTGGCTCTCCTCCCCCCCCCCCCCCGGAGGGCTGCGCCT...
1,AO090009000002,9682,14221,-1,AO090009000002_UP45,AO090009000002_DW45,AO090009000002_DEL_45bp_arms,9637,9682,14221,14266,45,45,90,TATTGATTCTATATATCTATATCTTGGATATACAACCTCGAATCC,GGTCTTTAAGCTCCGTCCAATATATTGACAGCCCGGATAGCCGCG,TATTGATTCTATATATCTATATCTTGGATATACAACCTCGAATCCG...
2,AO090009000003,15691,18301,-1,AO090009000003_UP45,AO090009000003_DW45,AO090009000003_DEL_45bp_arms,15646,15691,18301,18346,45,45,90,GGATATATATATTAACAATACAGTCGATTAAAGAAGATAGAATAA,CGAGATAGGATTTTTTACTTGCGGATCCAAGATGACTAAACCGAT,GGATATATATATTAACAATACAGTCGATTAAAGAAGATAGAATAAC...


In [9]:
repair_design_df = homology_arm_df.assign(
    deletion_name=lambda df: df['locus_tag'] + '_DEL',
    repair_design_name=lambda df: df['repair_oligo_name'],
    deleted_interval=lambda df: df['gene_start'].astype(str) + ':' + df['gene_end'].astype(str),
)[[
    'locus_tag',
    'deletion_name',
    'repair_design_name',
    'gene_start',
    'gene_end',
    'gene_strand',
    'deleted_interval',
    'upstream_arm_name',
    'upstream_arm_start',
    'upstream_arm_end',
    'upstream_arm_length',
    'downstream_arm_name',
    'downstream_arm_start',
    'downstream_arm_end',
    'downstream_arm_length',
    'repair_oligo_length',
    'repair_oligo',
]]

repair_design_df

,locus_tag,deletion_name,repair_design_name,gene_start,gene_end,gene_strand,deleted_interval,upstream_arm_name,upstream_arm_start,upstream_arm_end,upstream_arm_length,downstream_arm_name,downstream_arm_start,downstream_arm_end,downstream_arm_length,repair_oligo_length,repair_oligo
0,AO090009000001,AO090009000001_DEL,AO090009000001_DEL_45bp_arms,881,5162,1,881:5162,AO090009000001_UP45,836,881,45,AO090009000001_DW45,5162,5207,45,90,TCCCCTTCTGGCTCTCCTCCCCCCCCCCCCCCGGAGGGCTGCGCCT...
1,AO090009000002,AO090009000002_DEL,AO090009000002_DEL_45bp_arms,9682,14221,-1,9682:14221,AO090009000002_UP45,9637,9682,45,AO090009000002_DW45,14221,14266,45,90,TATTGATTCTATATATCTATATCTTGGATATACAACCTCGAATCCG...
2,AO090009000003,AO090009000003_DEL,AO090009000003_DEL_45bp_arms,15691,18301,-1,15691:18301,AO090009000003_UP45,15646,15691,45,AO090009000003_DW45,18301,18346,45,90,GGATATATATATTAACAATACAGTCGATTAAAGAAGATAGAATAAC...


In [10]:
primary_guide_df = (
    selected_guides
    .sort_values(['locus_tag', 'off_target_count', 'gc'])
    .groupby('locus_tag')
    .head(1)
    .loc[:, ['locus_tag', 'sgrna', 'pam', 'off_target_count']]
    .rename(columns={'sgrna': 'primary_sgrna'})
    .reset_index(drop=True)
)

crispr_deletion_design_df = repair_design_df.merge(primary_guide_df, on='locus_tag', how='left')
crispr_deletion_design_df

,locus_tag,deletion_name,repair_design_name,gene_start,gene_end,gene_strand,deleted_interval,upstream_arm_name,upstream_arm_start,upstream_arm_end,upstream_arm_length,downstream_arm_name,downstream_arm_start,downstream_arm_end,downstream_arm_length,repair_oligo_length,repair_oligo,primary_sgrna,pam,off_target_count
0,AO090009000001,AO090009000001_DEL,AO090009000001_DEL_45bp_arms,881,5162,1,881:5162,AO090009000001_UP45,836,881,45,AO090009000001_DW45,5162,5207,45,90,TCCCCTTCTGGCTCTCCTCCCCCCCCCCCCCCGGAGGGCTGCGCCT...,TCACATGGATAACAACAGAGGAG,TTTC,0
1,AO090009000002,AO090009000002_DEL,AO090009000002_DEL_45bp_arms,9682,14221,-1,9682:14221,AO090009000002_UP45,9637,9682,45,AO090009000002_DW45,14221,14266,45,90,TATTGATTCTATATATCTATATCTTGGATATACAACCTCGAATCCG...,TACGATCCCTGAAGATCCAGTAT,TTTA,0
2,AO090009000003,AO090009000003_DEL,AO090009000003_DEL_45bp_arms,15691,18301,-1,15691:18301,AO090009000003_UP45,15646,15691,45,AO090009000003_DW45,18301,18346,45,90,GGATATATATATTAACAATACAGTCGATTAAAGAAGATAGAATAAC...,ATCAAATCCAGATCAACACCTTC,TTTG,0


## Example ordering table

In [11]:
guide_order_df = selected_guides.assign(
    oligo_name=lambda df: df['locus_tag'] + '_sgRNA_' + (df.groupby('locus_tag').cumcount() + 1).astype(str),
    item_type='Cas12a crRNA spacer',
    sequence=lambda df: df['sgrna'],
)[['oligo_name', 'item_type', 'sequence', 'locus_tag', 'off_target_count']]

repair_oligo_order_df = homology_arm_df.assign(
    oligo_name=lambda df: df['repair_oligo_name'],
    item_type='repair oligo',
    sequence=lambda df: df['repair_oligo'],
)[['oligo_name', 'item_type', 'sequence', 'locus_tag', 'repair_oligo_length']]

order_table = pd.concat(
    [guide_order_df, repair_oligo_order_df],
    ignore_index=True,
)
order_table

,oligo_name,item_type,sequence,locus_tag,off_target_count,repair_oligo_length
0,AO090009000001_sgRNA_1,Cas12a crRNA spacer,TCACATGGATAACAACAGAGGAG,AO090009000001,0.0,NaN
1,AO090009000001_sgRNA_2,Cas12a crRNA spacer,TGATCTTCATCTCCTGTCTCAGA,AO090009000001,0.0,NaN
2,AO090009000002_sgRNA_1,Cas12a crRNA spacer,TACGATCCCTGAAGATCCAGTAT,AO090009000002,0.0,NaN
3,AO090009000002_sgRNA_2,Cas12a crRNA spacer,GAGTTTTGCATCGAGCTGATGAA,AO090009000002,0.0,NaN
4,AO090009000003_sgRNA_1,Cas12a crRNA spacer,ATCAAATCCAGATCAACACCTTC,AO090009000003,0.0,NaN
5,AO090009000003_sgRNA_2,Cas12a crRNA spacer,AACAAAGACAGCTAGTAAGGGTT,AO090009000003,0.0,NaN
6,AO090009000001_DEL_45bp_arms,repair oligo,TCCCCTTCTGGCTCTCCTCCCCCCCCCCCCCCGGAGGGCTGCGCCT...,AO090009000001,NaN,90.0
7,AO090009000002_DEL_45bp_arms,repair oligo,TATTGATTCTATATATCTATATCTTGGATATACAACCTCGAATCCG...,AO090009000002,NaN,90.0
8,AO090009000003_DEL_45bp_arms,repair oligo,GGATATATATATTAACAATACAGTCGATTAAAGAAGATAGAATAAC...,AO090009000003,NaN,90.0


## Eurofins-ready oligo table

In [12]:
eurofins_order_df = order_table.assign(
    Name=lambda df: df['oligo_name'],
    Sequence=lambda df: df['sequence'],
    Scale='25nm',
    Purification='Desalted',
    Form='Dry',
    Comments=lambda df: df['item_type'] + ' | ' + df['locus_tag'],
)[['Name', 'Sequence', 'Scale', 'Purification', 'Form', 'Comments']]

eurofins_order_df

,Name,Sequence,Scale,Purification,Form,Comments
0,AO090009000001_sgRNA_1,TCACATGGATAACAACAGAGGAG,25nm,Desalted,Dry,Cas12a crRNA spacer | AO090009000001
1,AO090009000001_sgRNA_2,TGATCTTCATCTCCTGTCTCAGA,25nm,Desalted,Dry,Cas12a crRNA spacer | AO090009000001
2,AO090009000002_sgRNA_1,TACGATCCCTGAAGATCCAGTAT,25nm,Desalted,Dry,Cas12a crRNA spacer | AO090009000002
3,AO090009000002_sgRNA_2,GAGTTTTGCATCGAGCTGATGAA,25nm,Desalted,Dry,Cas12a crRNA spacer | AO090009000002
4,AO090009000003_sgRNA_1,ATCAAATCCAGATCAACACCTTC,25nm,Desalted,Dry,Cas12a crRNA spacer | AO090009000003
5,AO090009000003_sgRNA_2,AACAAAGACAGCTAGTAAGGGTT,25nm,Desalted,Dry,Cas12a crRNA spacer | AO090009000003
6,AO090009000001_DEL_45bp_arms,TCCCCTTCTGGCTCTCCTCCCCCCCCCCCCCCGGAGGGCTGCGCCT...,25nm,Desalted,Dry,repair oligo | AO090009000001
7,AO090009000002_DEL_45bp_arms,TATTGATTCTATATATCTATATCTTGGATATACAACCTCGAATCCG...,25nm,Desalted,Dry,repair oligo | AO090009000002
8,AO090009000003_DEL_45bp_arms,GGATATATATATTAACAATACAGTCGATTAAAGAAGATAGAATAAC...,25nm,Desalted,Dry,repair oligo | AO090009000003


This notebook is meant as a compact Cas12a design example. A fuller workflow could extend this by:

- exporting the Eurofins table directly to CSV for ordering
- adding a fuller donor-construction strategy around the repair oligos
- generating validation primers for edited strains
- creating build-ready plate plans or project folders
- scaling the workflow across additional *A. oryzae* chromosomes or other fungi
